In [24]:
import json
import os
import pandas as pd
import numpy as np
import cv2

DATA_DIR = '../input/competitions/cassava-leaf-disease-classification/'

with open(os.path.join(DATA_DIR, 'label_num_to_disease_map.json')) as f:
    disease_map = json.load(f)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# 숫자로 된 label을 실제 질병 이름으로 바꾼 열을 하나 더 추가
df['disease_name'] = df['label'].astype(str).map(disease_map)

In [25]:
from sklearn.model_selection import train_test_split

#클래스 불균형이 발생했으므로 label의 개수를 동일하게 split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"학습용 데이터 개수: {len(train_df)}장")
print(f"테스트용 데이터 개수: {len(test_df)}장")

학습용 데이터 개수: 17117장
테스트용 데이터 개수: 4280장


In [26]:
import torch
from torch.utils.data import Dataset

class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform 
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_name = self.df.loc[index, 'image_id']
        label = self.df.loc[index, 'label']

        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
        
        if self.transform is not None:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [27]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),  # 좌우 반전
    transforms.RandomRotation(10),           # 살짝 회전
    transforms.ToTensor(),
])

In [28]:
from torch.utils.data import DataLoader

train_dataset = CassavaDataset(df=train_df, image_dir=DATA_DIR+'train_images', transform=transform)
test_dataset = CassavaDataset(df=test_df, image_dir=DATA_DIR+'train_images', transform=transform)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False,
    num_workers=2
)

In [29]:
from torchvision import models
import torch.nn as nn

# ResNet 모델 load
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

In [30]:
# 출력층 개수 변경

n_features = model.fc.in_features
model.fc = nn.Linear(n_features, 5)

In [31]:
# 모델 GPU로 전송

if torch.cuda.is_available():
    print("yes")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

yes


In [32]:
#손실함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-6)

In [33]:
epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)

        #오차역전파
        loss.backward()

        #가중치 수정
        optimizer.step()

        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] 완료! - 평균 오답 점수(Loss): {epoch_loss:.4f}")
print("학습 완료")

Epoch [1/15] 완료! - 평균 오답 점수(Loss): 0.6739
Epoch [2/15] 완료! - 평균 오답 점수(Loss): 0.4589
Epoch [3/15] 완료! - 평균 오답 점수(Loss): 0.3829
Epoch [4/15] 완료! - 평균 오답 점수(Loss): 0.3204
Epoch [5/15] 완료! - 평균 오답 점수(Loss): 0.2697
Epoch [6/15] 완료! - 평균 오답 점수(Loss): 0.2204
Epoch [7/15] 완료! - 평균 오답 점수(Loss): 0.1780
Epoch [8/15] 완료! - 평균 오답 점수(Loss): 0.1408
Epoch [9/15] 완료! - 평균 오답 점수(Loss): 0.1181
Epoch [10/15] 완료! - 평균 오답 점수(Loss): 0.1000
Epoch [11/15] 완료! - 평균 오답 점수(Loss): 0.0843
Epoch [12/15] 완료! - 평균 오답 점수(Loss): 0.0817
Epoch [13/15] 완료! - 평균 오답 점수(Loss): 0.0719
Epoch [14/15] 완료! - 평균 오답 점수(Loss): 0.0648
Epoch [15/15] 완료! - 평균 오답 점수(Loss): 0.0634
학습 완료


In [34]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_labels, all_preds)
print(f"Test 정확도: {final_accuracy * 100:.2f}%")

print("\n Classification Report")

disease_names = ['CBB (0)', 'CBSD (1)', 'CGM (2)', 'CMD (3)', 'Healthy (4)']
print(classification_report(all_labels, all_preds, target_names=disease_names))

Test 정확도: 82.17%

 Classification Report
              precision    recall  f1-score   support

     CBB (0)       0.50      0.57      0.53       217
    CBSD (1)       0.77      0.61      0.68       438
     CGM (2)       0.71      0.68      0.70       477
     CMD (3)       0.93      0.94      0.93      2632
 Healthy (4)       0.59      0.64      0.61       516

    accuracy                           0.82      4280
   macro avg       0.70      0.69      0.69      4280
weighted avg       0.82      0.82      0.82      4280

